# PaceAlgo ML — Notebook 05: LightGBM Training

**Der entscheidende Schritt.** Hier trainieren wir das erste echte ML-Modell und sehen ob PaceAlgo wirklich eine Edge generieren kann.

**Training-Setup:**
- **Symbole für Training:** BTC, ETH, EUR, USDJPY, XAU (5 Symbole × 2 TFs = 10 streams)
- **Hold-out (NICHT für Training):** SOL, GBPUSD (Dev Hold-out für Generalisierung)
- **R-Value:** 1.5 (kleinster, höchste WR-Baseline)
- **Walk-Forward Split:**
  - Train:      2020-01-01 → 2024-01-01
  - Validation: 2024-01-01 → 2024-07-01 (Early-Stopping + Threshold-Tuning)
  - Test:       2024-07-01 → 2026-05-01 (echtes OOS)
- **Hold-out Symbol Test:** Modell auf SOL+GBPUSD (komplett unbekannte Märkte)

**Modell:** LightGBM Binary Classifier (P(TP-first-hit))
- max_depth=3, num_leaves=7, 30-100 trees (early stopping)
- is_unbalance=True (class weight balanced)
- L2 regularization
- Pine-Budget-kompatibel

**Ziel:** PF > 1.5 auf Test-Set bei vernünftiger Trade-Frequenz.

## 1. Environment Setup

In [ ]:
import sys, os
from pathlib import Path
from datetime import datetime, timezone

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    MODELS_DIR = ARTIFACTS / 'models'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

    # Restore processed data from Drive backup if local is empty
    local_count = len(list(DATA_PROCESSED.glob('*.parquet')))
    drive_count = len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0
    print(f'Local files: {local_count}, Drive backup: {drive_count}')
    if local_count < drive_count:
        print('Restoring from Drive backup...')
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
        print(f'Files now in /content/processed/: {len(list(DATA_PROCESSED.glob("*.parquet")))}')
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_PROCESSED, ARTIFACTS_MODELS, ARTIFACTS_REPORTS
    MODELS_DIR = ARTIFACTS_MODELS
    REPORTS_DIR = ARTIFACTS_REPORTS

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]
print(f'Data:    {DATA_PROCESSED}')
print(f'Models:  {MODELS_DIR}')
print(f'Reports: {REPORTS_DIR}')

In [ ]:
!pip install -q lightgbm shap 2>&1 | tail -1

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib

from core.config import (
    CRYPTO_SYMBOLS, FX_SYMBOLS, METAL_SYMBOLS,
    PRIMARY_TIMEFRAMES,
    DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END,
    LGBM_DEFAULTS,
)
from core.train import (
    stack_symbols, get_feature_columns,
    walk_forward_split, binary_label_for_long,
    train_lgbm, evaluate_classifier,
    trading_metrics_from_predictions, sweep_threshold,
    load_features_and_labels,
)

ALL_SYMBOLS = CRYPTO_SYMBOLS + FX_SYMBOLS + METAL_SYMBOLS
R_VALUE = 1.5
print(f'All symbols:  {ALL_SYMBOLS}')
print(f'Dev hold-out: {DEV_HOLDOUT_SYMBOLS}')
print(f'Training symbols: {[s for s in ALL_SYMBOLS if s not in DEV_HOLDOUT_SYMBOLS]}')
print(f'R value: {R_VALUE}')
print(f'Train end: {TRAIN_END}, Val end: {VAL_END}')

## 2. Build Training Pool (stacked across symbols/TFs)

In [ ]:
df = stack_symbols(
    DATA_PROCESSED,
    symbols=ALL_SYMBOLS,
    tfs=PRIMARY_TIMEFRAMES,
    R=R_VALUE,
    drop_holdout=DEV_HOLDOUT_SYMBOLS,
)
print(f'Stacked DataFrame: {df.shape}')
print(f'Date range: {df.index.min()} -> {df.index.max()}')
print(f'\nLabel distribution (triple barrier):')
print(df["label"].value_counts().sort_index())
print(f'\nPer-symbol row counts:')
print(df.groupby(["symbol", "timeframe"]).size().unstack(fill_value=0))

## 3. Walk-Forward Split

In [ ]:
feature_cols = get_feature_columns(df)
print(f'Feature columns ({len(feature_cols)}): {feature_cols[:5]} ...')

# Drop rows with any NaN in features
df_clean = df.dropna(subset=feature_cols)
print(f'After NaN drop: {len(df_clean):,} of {len(df):,} rows ({len(df_clean)/len(df)*100:.1f}%)')

train_df, val_df, test_df = walk_forward_split(df_clean, TRAIN_END, VAL_END)
print(f'\nTrain:      {len(train_df):>10,} rows  {train_df.index.min()} -> {train_df.index.max()}')
print(f'Validation: {len(val_df):>10,} rows  {val_df.index.min()} -> {val_df.index.max()}')
print(f'Test:       {len(test_df):>10,} rows  {test_df.index.min()} -> {test_df.index.max()}')

# Binary labels for LightGBM
X_train = train_df[feature_cols]; y_train = binary_label_for_long(train_df['label'])
X_val = val_df[feature_cols]; y_val = binary_label_for_long(val_df['label'])
X_test = test_df[feature_cols]; y_test = binary_label_for_long(test_df['label'])

print(f'\nPositive rate — train: {y_train.mean():.3f}, val: {y_val.mean():.3f}, test: {y_test.mean():.3f}')

## 4. Train LightGBM

In [ ]:
params = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'num_leaves':       7,
    'max_depth':        3,
    'min_data_in_leaf': 500,    # heavier reg for big dataset
    'learning_rate':    0.05,
    'num_iterations':   100,
    'lambda_l2':        1.0,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'is_unbalance':     True,
    'verbose':          -1,
    'n_jobs':           -1,
}
print('Training LightGBM...')
model = train_lgbm(X_train, y_train, X_val, y_val, params=params, early_stopping_rounds=15)
print(f'\nBest iteration: {model.best_iteration}')
print(f'Number of trees in final model: {model.num_trees()}')

## 5. Classifier Metrics — Train / Val / Test

In [ ]:
for name, X, y in [('TRAIN', X_train, y_train), ('VAL', X_val, y_val), ('TEST', X_test, y_test)]:
    m = evaluate_classifier(model, X, y, threshold=0.5)
    print(f'\n=== {name} ===')
    print(f'  AUC:        {m["auc"]:.4f}')
    print(f'  LogLoss:    {m["log_loss"]:.4f}')
    print(f'  Accuracy:   {m["accuracy"]:.4f}  (positive rate true: {m["actual_positive_rate"]:.3f})')
    print(f'  Precision:  {m["precision"]:.4f}')
    print(f'  Recall:     {m["recall"]:.4f}')
    print(f'  F1:         {m["f1"]:.4f}')
    print(f'  Confusion:  TP={m["tp"]}, FP={m["fp"]}, TN={m["tn"]}, FN={m["fn"]}')

## 6. Trading Metrics — Threshold Sweep on VAL

Probieren verschiedene Confidence-Thresholds und schauen wo der Sweet Spot zwischen Trade-Häufigkeit und Profit Factor ist.

In [ ]:
val_proba = model.predict(X_val)
sweep_val = sweep_threshold(val_df['label'], val_proba, tp_R=R_VALUE)
print('Threshold sweep on VALIDATION set:')
display(sweep_val.round(4))

# Pick best threshold by PF (with min 1000 trades to avoid noise)
candidates = sweep_val[sweep_val['n_trades'] >= 1000]
if len(candidates) > 0:
    best = candidates.loc[candidates['profit_factor'].idxmax()]
    BEST_THRESHOLD = float(best['threshold'])
    print(f'\nBest threshold (from VAL): {BEST_THRESHOLD} (PF={best["profit_factor"]:.3f}, WR={best["win_rate"]:.3f}, n={int(best["n_trades"])})')
else:
    BEST_THRESHOLD = 0.55
    print(f'\nNo threshold with >=1000 trades — falling back to {BEST_THRESHOLD}')

## 7. OOS Performance on TEST Set

Apply the chosen threshold to the test set — diese Zahl ist die ehrliche Edge-Schätzung.

In [ ]:
test_proba = model.predict(X_test)
sweep_test = sweep_threshold(test_df['label'], test_proba, tp_R=R_VALUE)
print('Threshold sweep on TEST set (out-of-sample):')
display(sweep_test.round(4))

test_metrics = trading_metrics_from_predictions(test_df['label'], test_proba, BEST_THRESHOLD, tp_R=R_VALUE)
print(f'\nAt chosen threshold {BEST_THRESHOLD}:')
for k, v in test_metrics.items():
    print(f'  {k:20s}: {v}')

# Compare to random-entry baseline
baseline_pf = 0.983  # from notebook 04 high_vol R=1.5
print(f'\nBaseline (random entry) PF: {baseline_pf:.3f}')
print(f'Model PF on TEST:           {test_metrics["profit_factor"]:.3f}')
if test_metrics['profit_factor'] > 1.5:
    print('  -> EDGE CONFIRMED (PF > 1.5). Proceed to backtest.')
elif test_metrics['profit_factor'] > 1.2:
    print('  -> Modest edge. May improve with feature engineering or tuning.')
else:
    print('  -> No edge yet. Architecture revisit needed before proceeding.')

## 8. Hold-out Symbol Test (SOL, GBPUSD — never seen during training)

In [ ]:
holdout_rows = []
for sym in DEV_HOLDOUT_SYMBOLS:
    for tf in PRIMARY_TIMEFRAMES:
        ho_df = load_features_and_labels(DATA_PROCESSED, sym, tf, R_VALUE)
        if ho_df.empty:
            print(f'  SKIP {sym} {tf}: missing')
            continue
        ho_df = ho_df.dropna(subset=feature_cols)
        # Only test on the SAME OOS time window as test set
        ho_df = ho_df[ho_df.index >= pd.Timestamp(VAL_END).tz_localize('UTC')]
        if ho_df.empty:
            continue
        ho_proba = model.predict(ho_df[feature_cols])
        m = trading_metrics_from_predictions(ho_df['label'], ho_proba, BEST_THRESHOLD, tp_R=R_VALUE)
        m['symbol'] = sym
        m['tf'] = tf
        holdout_rows.append(m)
        print(f'  {sym} {tf}: n={m["n_trades"]}, WR={m["win_rate"]:.3f}, PF={m["profit_factor"]:.3f}, ExpR={m["expected_R"]:.4f}')

if holdout_rows:
    ho_summary = pd.DataFrame(holdout_rows)
    print('\nHold-out symbols summary:')
    display(ho_summary[['symbol','tf','n_trades','win_rate','profit_factor','expected_R','trade_rate']].round(4))

## 9. Feature Importance

In [ ]:
imp = pd.DataFrame({
    'feature': feature_cols,
    'gain':    model.feature_importance(importance_type='gain'),
    'split':   model.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

print('Top 20 features by gain:')
display(imp.head(20))

# Plot
fig, ax = plt.subplots(figsize=(9, 8))
imp_top = imp.head(20).iloc[::-1]
ax.barh(imp_top['feature'], imp_top['gain'], color='#3a7bd5')
ax.set_xlabel('Importance (gain)')
ax.set_title('LightGBM Feature Importance — Top 20')
plt.tight_layout()
plt.show()

## 10. Save Model + Report

In [ ]:
model_path = MODELS_DIR / f'lgbm_R{R_VALUE}.txt'
model.save_model(str(model_path))
print(f'Model saved: {model_path} ({model_path.stat().st_size / 1024:.1f} KB)')

# Save metadata (feature columns, threshold, R value)
import json
meta = {
    'feature_cols': feature_cols,
    'threshold': BEST_THRESHOLD,
    'R': R_VALUE,
    'sl_atr_mult': 1.0,
    'num_trees': model.num_trees(),
    'best_iteration': int(model.best_iteration) if model.best_iteration else model.num_trees(),
    'train_end': str(TRAIN_END),
    'val_end': str(VAL_END),
    'training_symbols': [s for s in ALL_SYMBOLS if s not in DEV_HOLDOUT_SYMBOLS],
    'holdout_symbols': DEV_HOLDOUT_SYMBOLS,
    'test_metrics': {k: float(v) if isinstance(v, (int, float)) else v for k, v in test_metrics.items()},
}
meta_path = MODELS_DIR / f'lgbm_R{R_VALUE}_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Meta saved: {meta_path}')

# Save full report
report_path = REPORTS_DIR / f'lgbm_R{R_VALUE}_report.parquet'
sweep_test.assign(set='test').to_parquet(report_path, compression='zstd')
print(f'Report saved: {report_path}')

---

## Nächster Schritt — Entscheidung basierend auf Test PF

| Test PF | Bedeutung | Aktion |
|---|---|---|
| > 1.7 | Strong edge | → Notebook 06: XGBoost-Vergleich + Notebook 09: Pine Export |
| 1.3-1.7 | Modest edge | → Mehr Feature Engineering, tuning, dann Pine Export |
| < 1.3 | Kein Edge | → Architektur-Review: Labeling, Features, Meta-Labels evaluieren |

Schick Test-PF + Hold-out-Results — wir entscheiden basierend auf den Zahlen.